# 04. Shor's algorithm: quantum order-finding for N = 15
Shor's algorithm factors N by finding the **order** r of a number a modulo N (the smallest r with $a^r \equiv 1 \pmod N$). Only that step is quantum. It uses phase estimation, with an inverse quantum Fourier transform. The rest is classical.

This notebook does N=15, a=7 on a simulator. The modular multiplication circuits for N=15 are hard-coded, following the approach in the Qiskit textbook, which I used as a reference. It does not scale to larger N.

In [1]:
import numpy as np
from math import gcd
from fractions import Fraction
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def qft_dagger(n):
    qc = QuantumCircuit(n)
    for q in range(n//2): qc.swap(q, n-q-1)
    for j in range(n):
        for m in range(j): qc.cp(-np.pi/float(2**(j-m)), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc

def c_amod15(a, power):
    """Controlled multiplication by a^power mod 15 (hard-coded for N=15)."""
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]: U.swap(0,1); U.swap(1,2); U.swap(2,3)
        if a in [7, 8]:  U.swap(2,3); U.swap(1,2); U.swap(0,1)
        if a in [4, 11]: U.swap(0,2); U.swap(1,3)
        if a in [7, 11, 13]:
            for q in range(4): U.x(q)
    g = U.to_gate(); g.name = f"{a}^{power} mod 15"
    return g.control()

In [2]:
N, a, n_count = 15, 7, 8
qc = QuantumCircuit(n_count+4, n_count)
qc.h(range(n_count))
qc.x(n_count)                                   # target register starts in |0001>
for q in range(n_count):
    qc.append(c_amod15(a, 2**q), [q] + [n_count+i for i in range(4)])
qc.append(qft_dagger(n_count), range(n_count))
qc.measure(range(n_count), range(n_count))

sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
top = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:4]
for bits, shots in top:
    print(f"{bits}  phase={int(bits,2)/2**n_count:.4f}  shots={shots}")

01000000  phase=0.2500  shots=271
00000000  phase=0.0000  shots=257
10000000  phase=0.5000  shots=255
11000000  phase=0.7500  shots=241


## Classical post-processing
Each measured phase is close to $s/r$. A continued-fraction step (`limit_denominator`) recovers the candidate order r. Then $\gcd(a^{r/2}\pm1, N)$ gives the factors. Phase 0 gives r=1, which is useless, so we try the other outcomes.

In [3]:
factors = set()
for bits, shots in counts.items():
    phase = int(bits,2)/2**n_count
    r = Fraction(phase).limit_denominator(N).denominator
    if r % 2 == 0 and pow(a, r, N) == 1:
        for f in (gcd(pow(a, r//2) - 1, N), gcd(pow(a, r//2) + 1, N)):
            if f not in (1, N): factors.add(f)
print(f"order r found from measurements; factors of {N}:", sorted(factors))

order r found from measurements; factors of 15: [3, 5]
